# Monitor Fintech Job Runs

**Objective:** Inspect operational history for
fintech_credit_recovery_monitoring_job.

## Why?

**For our project:** Compare the repaired run with the successful dashboard
run and document the pipeline's execution history.

**For Data Engineering:** Practice querying operational system tables to
investigate failures, distinguish task attempts, and analyze execution durations.

In [ ]:
%sql
SHOW SCHEMAS IN system;

databaseName
access
ai
ai_gateway
alert
billing
compute
information_schema
lakeflow
lakeflow_pipeline_events_preview
mlflow


In [ ]:
%sql
SHOW TABLES IN system.lakeflow;

database,tableName,isTemporary
lakeflow,job_run_timeline,false
lakeflow,job_task_run_timeline,false
lakeflow,job_tasks,false
lakeflow,jobs,false
lakeflow,pipeline_update_timeline,false
lakeflow,pipelines,false
lakeflow,zerobus_ingest,false
lakeflow,zerobus_stream,false


#### 01 — Inspect Metadata Tables and Identify the Recovery Job

Inspect the table columns and retrieve the latest active job definition.
Use its workspace and job identifiers in subsequent monitoring queries.

In [ ]:
%sql
-- Job metadata columns
DESCRIBE TABLE system.lakeflow.jobs;

col_name,data_type,comment
account_id,string,The ID of the account this job belongs to.
workspace_id,string,The ID of the workspace this job belongs to.
job_id,string,The ID of the job. Only unique within a single workspace.
name,string,The user-supplied name of the job.
creator_id,string,The ID of the principal who created the job.
tags,"map<string,string>",The user-supplied custom tags associated with this job.
run_as,string,The ID of the user or service principal whose permissions are used for the job run.
change_time,timestamp,The time when the job was last modified. Timezone recorded as +00:00 (UTC).
delete_time,timestamp,The time when the job was deleted by the user. Timezone recorded as +00:00 (UTC).
description,string,The user-supplied description of the job. **Not populated for rows emitted before late August 2024.**


In [ ]:
%sql
-- Task execution columns
DESCRIBE TABLE system.lakeflow.job_task_run_timeline;

col_name,data_type,comment
account_id,string,The ID of the account this job belongs to.
workspace_id,string,The ID of the workspace this job belongs to.
job_id,string,The ID of the job. Only unique within a single workspace.
run_id,string,The ID of the task run.
period_start_time,timestamp,The start time for the task or for the time period. Timezone information is recorded at the end of the value with +00:00 representing UTC.
period_end_time,timestamp,The end time for the task or for the time period. Timezone information is recorded at the end of the value with +00:00 representing UTC.
task_key,string,The reference key for a task in a job. This key is only unique within a single job.
compute_ids,array<string>,An array containing the IDs of compute used by the job task.
result_state,string,The outcome of the job task run.
job_run_id,string,The ID of the job run. **Not populated for rows emitted before late August 2024.**


In [ ]:
%sql
-- Identify the recovery job
WITH latest_jobs AS (
    SELECT
        workspace_id,
        job_id,
        name,
        change_time,
        delete_time,
        ROW_NUMBER() OVER (
            PARTITION BY workspace_id, job_id
            ORDER BY change_time DESC
        ) AS row_num
    FROM system.lakeflow.jobs
    WHERE job_id = '<JOB_ID>'
)
SELECT
    workspace_id,
    job_id,
    name,
    change_time
FROM latest_jobs
WHERE row_num = 1
  AND delete_time IS NULL;

workspace_id,job_id,name,change_time
<WORKSPACE_ID>,<JOB_ID>,fintech_credit_recovery_monitoring_job,2026-11-03T13:34:43.950Z


#### 02 — Configure Monitoring Parameters

Select the workspace, job, and parent job run to inspect.
Start with the run used for the controlled failure and repair.

In [ ]:
dbutils.widgets.text("workspace_id", "<WORKSPACE_ID>")
dbutils.widgets.text("job_id", "<JOB_ID>")
dbutils.widgets.text("job_run_id", "<JOB_RUN_REPAIRED>")

#### 03 — Inspect Task Execution History

Join the current job definition to its task execution history.
Identify the failed application transformation and its successful repair.

In [ ]:
%sql
WITH latest_job AS (
    SELECT
        workspace_id,
        job_id,
        name,
        delete_time,
        ROW_NUMBER() OVER (
            PARTITION BY workspace_id, job_id
            ORDER BY change_time DESC
        ) AS row_num
    FROM system.lakeflow.jobs
    WHERE workspace_id = :workspace_id
      AND job_id = :job_id
)
SELECT
    j.name AS current_job_name,
    t.job_run_id,
    t.task_key,
    t.run_id AS task_run_id,
    t.period_start_time,
    t.period_end_time,
    t.result_state,
    t.termination_code
FROM system.lakeflow.job_task_run_timeline AS t
JOIN latest_job AS j
    ON t.workspace_id = j.workspace_id
   AND t.job_id = j.job_id
   AND j.row_num = 1
   AND j.delete_time IS NULL
WHERE t.workspace_id = :workspace_id
  AND t.job_id = :job_id
  AND t.job_run_id = :job_run_id
ORDER BY t.period_start_time, t.task_key, t.run_id;

current_job_name,job_run_id,task_key,task_run_id,period_start_time,period_end_time,result_state,termination_code
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,ingesting_applications,<RECORDED_TASK_RUN_01>,2026-11-10T13:55:29.963Z,2026-11-10T13:56:03.721Z,SUCCEEDED,SUCCESS
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,ingesting_customers,<RECORDED_TASK_RUN_02>,2026-11-10T13:55:29.965Z,2026-11-10T13:55:46.361Z,SUCCEEDED,SUCCESS
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,ingesting_installments,<RECORDED_TASK_RUN_03>,2026-11-10T13:55:29.971Z,2026-11-10T13:56:03.219Z,SUCCEEDED,SUCCESS
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,customers_installments_summary,<RECORDED_TASK_RUN_04>,2026-11-10T13:56:03.606Z,2026-11-10T13:56:20.523Z,SUCCEEDED,SUCCESS
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,customers_applications_report,<RECORDED_TASK_RUN_05>,2026-11-10T13:56:04.133Z,2026-11-10T13:56:18.271Z,SUCCEEDED,SUCCESS
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,customers_applications_state_report_iterator,<RECORDED_TASK_RUN_06>,2026-11-10T13:56:18.605Z,2026-11-10T13:57:24.354Z,SUCCEEDED,SUCCESS
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,customers_applications_state_report,<RECORDED_TASK_RUN_07>,2026-11-10T13:56:19.663Z,2026-11-10T13:56:31.759Z,SUCCEEDED,SUCCESS
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,checking_for_duplicates,<RECORDED_TASK_RUN_08>,2026-11-10T13:56:20.917Z,2026-11-10T13:56:21.990Z,SUCCEEDED,SUCCESS
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,dropping_duplicate_records,<RECORDED_TASK_RUN_09>,2026-11-10T13:56:22.206Z,2026-11-10T13:56:22.206Z,SKIPPED,SKIPPED
fintech_credit_recovery_monitoring_job,<JOB_RUN_FINAL>,transforming_customers_installments,<RECORDED_TASK_RUN_10>,2026-11-10T13:56:22.339Z,2026-11-10T13:56:34.181Z,SUCCEEDED,SUCCESS


#### 04 — Calculate Task Attempt Durations

Combine timeline slices for each task run and calculate its duration.
Keep separate task run IDs so the failed attempt and repair remain distinct.

In [ ]:
%sql
SELECT
    task_key,
    run_id AS task_run_id,
    MIN(period_start_time) AS task_start,
    MAX(period_end_time) AS task_end,
    ROUND(
        SUM(
            TIMESTAMPDIFF(
                MICROSECOND,
                period_start_time,
                period_end_time
            )
        ) / 1000000.0,
        3
    ) AS task_timeline_seconds,
    MAX_BY(result_state, period_end_time) AS result_state
FROM system.lakeflow.job_task_run_timeline
WHERE workspace_id = :workspace_id
  AND job_id = :job_id
  AND job_run_id = :job_run_id
GROUP BY
    workspace_id,
    job_id,
    job_run_id,
    task_key,
    run_id
ORDER BY task_start, task_key, task_run_id;

task_key,task_run_id,task_start,task_end,task_timeline_seconds,result_state
ingesting_applications,<RECORDED_TASK_RUN_01>,2026-11-10T13:55:29.963Z,2026-11-10T13:56:03.721Z,33.758,SUCCEEDED
ingesting_customers,<RECORDED_TASK_RUN_02>,2026-11-10T13:55:29.965Z,2026-11-10T13:55:46.361Z,16.396,SUCCEEDED
ingesting_installments,<RECORDED_TASK_RUN_03>,2026-11-10T13:55:29.971Z,2026-11-10T13:56:03.219Z,33.248,SUCCEEDED
customers_installments_summary,<RECORDED_TASK_RUN_04>,2026-11-10T13:56:03.606Z,2026-11-10T13:56:20.523Z,16.917,SUCCEEDED
customers_applications_report,<RECORDED_TASK_RUN_05>,2026-11-10T13:56:04.133Z,2026-11-10T13:56:18.271Z,14.138,SUCCEEDED
customers_applications_state_report_iterator,<RECORDED_TASK_RUN_06>,2026-11-10T13:56:18.605Z,2026-11-10T13:57:24.354Z,65.749,SUCCEEDED
customers_applications_state_report,<RECORDED_TASK_RUN_07>,2026-11-10T13:56:19.663Z,2026-11-10T13:56:31.759Z,12.096,SUCCEEDED
checking_for_duplicates,<RECORDED_TASK_RUN_08>,2026-11-10T13:56:20.917Z,2026-11-10T13:56:21.990Z,1.073,SUCCEEDED
dropping_duplicate_records,<RECORDED_TASK_RUN_09>,2026-11-10T13:56:22.206Z,2026-11-10T13:56:22.206Z,0.000,SKIPPED
transforming_customers_installments,<RECORDED_TASK_RUN_10>,2026-11-10T13:56:22.339Z,2026-11-10T13:56:34.181Z,11.842,SUCCEEDED


#### 05 — Inspect Job Completion Events

Review completion events across this job's runs.
Compare the original failure, successful repair, and final dashboard run.

In [ ]:
%sql
SELECT
    run_id AS job_run_id,
    period_end_time AS terminal_event_time,
    result_state,
    termination_code
FROM system.lakeflow.job_run_timeline
WHERE workspace_id = :workspace_id
  AND job_id = :job_id
  AND result_state IS NOT NULL
ORDER BY terminal_event_time DESC
LIMIT 50;

job_run_id,terminal_event_time,result_state,termination_code
<JOB_RUN_FINAL>,2026-11-10T13:58:09.620Z,SUCCEEDED,SUCCESS
<JOB_RUN_REPAIRED>,2026-11-03T20:57:46.509Z,SUCCEEDED,SUCCESS
<JOB_RUN_REPAIRED>,2026-11-03T20:47:01.859Z,ERROR,RUN_EXECUTION_ERROR
